# 📦 Notebook 1 — Data Pipeline

**Author:** Maulik Mathur | **HuggingFace:** [maulik78](https://huggingface.co/maulik78)

---

## Why This Notebook Exists

Every ML project lives or dies by its data. Before writing a single line of model code, I spent time understanding and engineering the dataset.

This notebook answers:
- Where does the data come from?
- What does raw Amazon data actually look like?
- Why do we need to clean it?
- How do we handle 2.9 million products without running out of RAM?
- Why not just sample randomly?

## The Data Engineering Story

```
RAW DATA                    AFTER THIS NOTEBOOK
─────────────────────────   ─────────────────────────────────────
2.9M messy Amazon items  →  820k clean, balanced, structured items
33% Automotive items     →  ~6% Automotive (corrected imbalance)
70% cheap items (<$30)   →  Balanced price distribution
String prices '$49.99'   →  Float prices 49.99
Part numbers everywhere  →  Clean text, no catalog IDs
Mixed weight units       →  Standardized to pounds
```

## Runtime
- **Platform:** Google Colab (CPU)
- **Time:** 3-5 hours
- **Cost:** Free

## Step 1 — Setup

In [ ]:
!pip install -q datasets huggingface_hub python-dotenv tqdm psutil

import os, sys, gc, pickle, random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.notebook import tqdm
import psutil

if not os.path.exists('/content/amazon-price-predictor'):
    !git clone https://github.com/maulik-04/amazon-price-predictor.git --quiet
sys.path.insert(0, '/content/amazon-price-predictor')

from pipeline.2_clean_data import clean_item
from pipeline.3_sample_data import compute_sampling_weights, split_dataset, TARGET_SAMPLE_SIZE, CATEGORY_MULTIPLIERS

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'), add_to_git_credential=True)

print(f'✅ Setup complete | RAM: {psutil.virtual_memory().available/1e9:.1f} GB available')

---
## Part 1 — Understand the Raw Data

Before building the pipeline, we inspect ONE small category (Appliances)
to understand the raw data structure without downloading everything.

In [ ]:
from datasets import load_dataset

# Load smallest category first to inspect structure
print('Loading Appliances (smallest category) to inspect structure...')
sample = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_meta_Appliances',
    split='full',
    trust_remote_code=True
)
print(f'Raw items: {len(sample):,}')
print(f'Columns: {sample.column_names}')

In [ ]:
# Inspect one raw item — this reveals ALL the cleaning problems
raw = sample[5]
print('RAW ITEM (before any cleaning):')
print('='*60)
for key, val in raw.items():
    print(f'{key:<15}: {str(val)[:150]}')
    print()
print('PROBLEMS:')
print('  1. price is string "$49.99" — needs float conversion')
print('  2. description is a list — needs joining')
print('  3. details is JSON string — needs parsing')
print('  4. part numbers like "B07XJ8C8F5" pollute ML features')
print('  5. weight has units like "1.5 pounds" — needs standardization')

In [ ]:
# Show price validity statistics
# KEY INSIGHT: Many items have missing/invalid prices — we can't use them
valid, invalid, out_of_range = [], 0, 0

for item in tqdm(sample, desc='Scanning prices'):
    try:
        price = float(str(item['price']).replace('$','').replace(',',''))
        if 0.5 <= price <= 999.49:
            valid.append(price)
        else:
            out_of_range += 1
    except:
        invalid += 1

total = len(sample)
print(f'Price analysis ({total:,} items):')
print(f'  Valid ($0.50-$999.49): {len(valid):,} ({100*len(valid)/total:.1f}%)')
print(f'  Invalid/missing:       {invalid:,} ({100*invalid/total:.1f}%)')
print(f'  Out of range:          {out_of_range:,} ({100*out_of_range/total:.1f}%)')
print(f'\nWhy $0.50-$999.49?')
print('  Under $0.50 = usually samples, shipping accessories, noise')
print('  Over $999 = industrial/commercial items, different patterns')
print('  We focus on consumer product price range')

---
## Part 2 — The Cleaning Logic

Our `clean_item()` function in `pipeline/2_clean_data.py` transforms
one raw item into a clean `Product` object or returns `None` to reject it.

In [ ]:
# Show before → after transformation for one item
for item in sample:
    cleaned = clean_item(item, 'Appliances')
    if cleaned:
        break

print('BEFORE (raw):')
print(f'  price:  {item["price"]}  ← string')
print(f'  text:   {(str(item["description"]) + str(item["features"]))[:200]}...')

print('\nAFTER (cleaned Product):')
print(f'  price:  {cleaned.price}  ← float')
print(f'  weight: {cleaned.weight} pounds')
print(f'  text:   {cleaned.text[:300]}...')
print(f'\nNOTE: Part numbers removed, text standardized, weight converted to pounds')

In [ ]:
# Run cleaning on all Appliances — see filtering impact
cleaned_items = [clean_item(item, 'Appliances') for item in tqdm(sample, desc='Cleaning')]
cleaned_items = [p for p in cleaned_items if p is not None]

print(f'Cleaning results:')
print(f'  {len(sample):,} raw → {len(cleaned_items):,} clean ({100*len(cleaned_items)/len(sample):.1f}% kept)')
print(f'  {len(sample)-len(cleaned_items):,} items rejected (bad price, too short, etc.)')

del sample
gc.collect()

---
## Part 3 — Load All 8 Categories

**Memory Strategy:**
Each category (1-5GB) is loaded → cleaned → saved to Drive → deleted from RAM.
Peak RAM stays under 4GB even though total raw data is 15GB+.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/price_predictor_data'
os.makedirs(DRIVE_DIR, exist_ok=True)

CATEGORIES = [
    'Automotive', 'Electronics', 'Office_Products',
    'Tools_and_Home_Improvement', 'Cell_Phones_and_Accessories',
    'Toys_and_Games', 'Appliances', 'Musical_Instruments',
]

saved = [f.replace('.pkl','') for f in os.listdir(DRIVE_DIR) if f.endswith('.pkl')]
print(f'Already saved: {saved}')

In [ ]:
# Load, clean, save each category one at a time
for category in CATEGORIES:
    if category in saved:
        print(f'⏭️  {category} — already on Drive')
        continue
    
    print(f'\n🔄 {category} | RAM: {psutil.virtual_memory().used/1e9:.1f} GB')
    try:
        raw = load_dataset('McAuley-Lab/Amazon-Reviews-2023', f'raw_meta_{category}', split='full', trust_remote_code=True)
        products = [p for p in (clean_item(item, category) for item in tqdm(raw, desc='Cleaning')) if p]
        print(f'  {len(raw):,} raw → {len(products):,} clean')
        
        with open(f'{DRIVE_DIR}/{category}.pkl', 'wb') as f:
            pickle.dump(products, f)
        saved.append(category)
        del raw, products
        gc.collect()
        print(f'  ✅ Saved | RAM: {psutil.virtual_memory().used/1e9:.1f} GB')
    except Exception as e:
        print(f'  ❌ {e}')

print(f'\n✅ All {len(saved)} categories saved to Drive')

---
## Part 4 — Weighted Sampling

**Two-Pass Strategy** to stay under RAM limits:
- Pass 1: Load only prices + categories (~200MB for 2.9M items)
- Pass 2: Load only the 820k selected items

In [ ]:
# PASS 1: Index all items (prices + categories only)
print('Pass 1: Building lightweight index...')
all_prices, all_categories, all_indices = [], [], []

for fname in sorted(os.listdir(DRIVE_DIR)):
    if not fname.endswith('.pkl'):
        continue
    cat_name = fname.replace('.pkl','')
    with open(f'{DRIVE_DIR}/{fname}', 'rb') as f:
        items = pickle.load(f)
    for i, item in enumerate(items):
        all_prices.append(item.price)
        all_categories.append(cat_name)
        all_indices.append((fname, i))
    del items
    gc.collect()
    print(f'  {cat_name}: {len(all_prices):,} total')

print(f'\nTotal: {len(all_prices):,} items | RAM: {psutil.virtual_memory().used/1e9:.1f} GB')
print('Notice: RAM is low because we only loaded prices, not full item data')

In [ ]:
# Calculate weights and select 820k items
prices_arr     = np.array(all_prices, dtype=float)
categories_arr = np.array(all_categories)

# Normalize to [0,1] then square — aggressively promotes expensive items
norm    = (prices_arr - prices_arr.min()) / (prices_arr.max() - prices_arr.min() + 1e-9)
weights = norm ** 2

print('Applying category multipliers:')
for cat, mult in CATEGORY_MULTIPLIERS.items():
    mask = categories_arr == cat
    weights[mask] *= mult
    print(f'  {cat:<35} ×{mult} ({mask.sum():,} items)')

weights = weights / weights.sum()  # normalize to probability distribution

np.random.seed(42)
selected = np.random.choice(len(all_prices), size=TARGET_SAMPLE_SIZE, replace=False, p=weights)

print(f'\nSelected {len(selected):,} items. Distribution after sampling:')
cat_counts = Counter([all_categories[i] for i in selected])
for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
    pct = count/len(selected)*100
    print(f'  {cat:<35} {count:>8,}  ({pct:.1f}%)')

del prices_arr, categories_arr, norm, weights
gc.collect()

In [ ]:
# PASS 2: Load only selected items
needed_by_file = defaultdict(set)
for idx in selected:
    fname, pos = all_indices[idx]
    needed_by_file[fname].add(pos)

final_products = []
for fname in sorted(needed_by_file.keys()):
    with open(f'{DRIVE_DIR}/{fname}', 'rb') as f:
        items = pickle.load(f)
    final_products.extend([items[pos] for pos in needed_by_file[fname]])
    del items
    gc.collect()
    print(f'  {fname.replace(".pkl","")}: loaded {len(needed_by_file[fname]):,} | RAM: {psutil.virtual_memory().used/1e9:.1f} GB')

print(f'\nTotal: {len(final_products):,} products')

In [ ]:
# Deduplicate
print(f'Before dedup: {len(final_products):,}')
random.seed(42)
random.shuffle(final_products)

seen = set()
final_products = [p for p in tqdm(final_products, desc='Title dedup') if not (p.title in seen or seen.add(p.title))]
print(f'After title dedup: {len(final_products):,}')

seen = set()
final_products = [p for p in tqdm(final_products, desc='Text dedup') if not (p.text in seen or seen.add(p.text))]
print(f'After text dedup:  {len(final_products):,}')
del seen
gc.collect()

In [ ]:
# Visualize final dataset
prices  = [p.price    for p in final_products]
cats    = [p.category for p in final_products]
lengths = [len(p.text) for p in final_products]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Final Dataset — 820k Amazon Products', fontsize=14)

axes[0].hist(prices, bins=range(0,1000,20), color='orange', rwidth=0.8)
axes[0].set_title(f'Price Distribution\nMean: ${sum(prices)/len(prices):.2f}')
axes[0].set_xlabel('Price ($)')

cat_counts = Counter(cats)
axes[1].bar(range(len(cat_counts)), list(cat_counts.values()), color='steelblue')
axes[1].set_xticks(range(len(cat_counts)))
axes[1].set_xticklabels([c.replace('_','\n') for c in cat_counts.keys()], fontsize=7, rotation=45)
axes[1].set_title('Category Distribution')

axes[2].hist(lengths, bins=50, color='lightgreen', rwidth=0.8)
axes[2].set_title(f'Text Length\nMean: {sum(lengths)/len(lengths):.0f} chars')
axes[2].set_xlabel('Characters')

plt.tight_layout()
plt.savefig('dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Split and push to HuggingFace
train, val, test = split_dataset(final_products)

from datasets import Dataset, DatasetDict

def to_hf(products):
    return Dataset.from_list([{'title':p.title,'category':p.category,'price':p.price,'text':p.text,'weight':p.weight,'summary':p.summary} for p in products])

USERNAME = 'maulik78'

print('Pushing full dataset...')
DatasetDict({'train':to_hf(train),'val':to_hf(val),'test':to_hf(test)}).push_to_hub(f'{USERNAME}/items_raw_full')
print(f'✅ huggingface.co/datasets/{USERNAME}/items_raw_full')

print('\nPushing lite dataset...')
DatasetDict({'train':to_hf(train[:20000]),'val':to_hf(val[:1000]),'test':to_hf(test[:1000])}).push_to_hub(f'{USERNAME}/items_raw_lite')
print(f'✅ huggingface.co/datasets/{USERNAME}/items_raw_lite')

print('\n'+'='*55)
print('✅ NOTEBOOK 1 COMPLETE — DATA PIPELINE')
print('='*55)
print('\n➡️  Next: Notebook 2 — Model Comparison')